In [1]:
from DataLoader import DataLoader 
from pathlib import Path 
import numpy as np 
import SimpleITK as sitk 

root = r"C:\Users\20202310\Desktop\MSc scriptie\MSc_Graduation_Project\data\LUNDPROBE\ExtendedSamples\development" 
rootpath = Path(root) 
subjects = sorted([p.name for p in rootpath.iterdir() if p.is_dir()]) 
print(subjects)


for subject_nr in range(len(subjects)):
    data = DataLoader(
        parentfolder=root,
        subject_nr=subject_nr,
        volume_of_interest="CTVT",
        verbose=True,
    )

    # Determine output path first
    if data.volume_of_interest == "CTVT":
        save_path = (
            data.subjectfolder
            / "observerData"
            / "mask_CTVT_427_step2_STAPLEConsensus.nii.gz"
        )
    else:
        save_path = (
            data.subjectfolder
            / "observerData"
            / "mask_Rectum_step2_STAPLEConsensus.nii.gz"
        )

    # Skip if STAPLE consensus already exists
    if save_path.exists():
        print(f"STAPLE consensus already exists for {save_path}")
        continue

    # Load observer recontours
    observer_masks = data.load_recontours()

    # Convert NumPy masks to SimpleITK images
    sitk_observer_masks = []

    for mask in observer_masks:
        mask_uint8 = mask.astype(np.uint8)
        mask_itk = sitk.GetImageFromArray(mask_uint8)
        sitk_observer_masks.append(mask_itk)

    # Run STAPLE
    staple_filter = sitk.STAPLEImageFilter()
    staple_itk = staple_filter.Execute(sitk_observer_masks)

    # Convert result back to NumPy
    staple_prob = sitk.GetArrayFromImage(staple_itk)

    # Threshold probability map to get binary consensus mask
    staple_gt = staple_prob >= 0.5

    print("STAPLE probability map shape:", staple_prob.shape)
    print("STAPLE binary mask shape:", staple_gt.shape)
    print("STAPLE min/max probability:", staple_prob.min(), staple_prob.max())
    print("Number of foreground voxels:", staple_gt.sum())

    # Convert consensus mask to SimpleITK image
    staple_itk_bin = sitk.GetImageFromArray(staple_gt.astype(np.uint8))

    # Copy spacing/origin/direction from existing ground truth
    if data.volume_of_interest == "CTVT":
        reference_path = data.subjectfolder / "mask_CTVT_427.nii.gz"
    else:
        reference_path = data.subjectfolder / "mask_Rectum.nii.gz"

    reference_img = sitk.ReadImage(str(reference_path))
    staple_itk_bin.CopyInformation(reference_img)

    sitk.WriteImage(staple_itk_bin, str(save_path))

    print(f"Saved STAPLE consensus to:\n{save_path}")

['newAcq_050f229dc2bdb64c', 'newAcq_0b4940fa31a1d650', 'newAcq_0cc559a8bd82a14a', 'newAcq_1b911d6cb2348f30', 'newAcq_1e0f8b9b01ce5f0b', 'newAcq_250d6075dd465a1a', 'newAcq_433a8d44fddd5b7f', 'newAcq_47ceabdbca398517', 'newAcq_486b7494ee9d71e7', 'newAcq_4a136e8fe320bd13']
Loaded subject newAcq_050f229dc2bdb64c with volume of interest 'CTVT'
Image shape: (88, 1024, 1024), Mask shape: (88, 1024, 1024), Uncertainty map shape: (88, 1024, 1024), Ground truth shape: (88, 1024, 1024)
Image spacing (z, y, x): (2.5, 0.46880000829696655, 0.46880000829696655)
STAPLE consensus already exists for C:\Users\20202310\Desktop\MSc scriptie\MSc_Graduation_Project\data\LUNDPROBE\ExtendedSamples\development\newAcq_050f229dc2bdb64c\MR_StorT2\observerData\mask_CTVT_427_step2_STAPLEConsensus.nii.gz
Loaded subject newAcq_0b4940fa31a1d650 with volume of interest 'CTVT'
Image shape: (88, 1024, 1024), Mask shape: (88, 1024, 1024), Uncertainty map shape: (88, 1024, 1024), Ground truth shape: (88, 1024, 1024)
Image s

In [2]:
import numpy as np
import matplotlib.pyplot as plt
from ipywidgets import interact, IntSlider, Checkbox
import matplotlib.patches as mpatches
from skimage.measure import find_contours


def plot_consensus_interactive(
    data,
    figsize=(7, 7),
    zoom_fraction=0.30,
):

    img = np.asarray(data.img)
    gt = np.asarray(data.gt).astype(bool)
    consensus = np.asarray(data.consensus_recontour).astype(bool)

    recontours = [
        np.asarray(r).astype(bool)
        for r in data.observer_recontours
    ]

    observer_names = getattr(
        data,
        "observer_names",
        [f"{i+1}" for i in range(len(recontours))]
    )

    n_slices = img.shape[0]

    color_gt = "#7fc97f"
    color_consensus = "#4f8cff"

    observer_colors = [
        "#ffcccc",
        "#ff8a8a",
        "#e63946",
        "#9d0208",
    ]

    def draw_contour(ax, mask, color, linewidth=2.0, zorder=1):
        contours = find_contours(mask.astype(float), level=0.5)

        for contour in contours:
            ax.plot(
                contour[:, 1],
                contour[:, 0],
                color=color,
                linewidth=linewidth,
                zorder=zorder,
            )

    def _plot(
        slice_idx,
        zoom,
        show_ground_truth,
        show_consensus,
        show_observer_B,
        show_observer_C,
        show_observer_D,
        show_observer_E,
    ):

        z = slice_idx
        H, W = img[z].shape

        show_observers = [
            show_observer_B,
            show_observer_C,
            show_observer_D,
            show_observer_E,
        ]

        if zoom:
            fg = gt[z] | consensus[z]

            for obs_seg, show_obs in zip(recontours, show_observers):
                if show_obs:
                    fg |= obs_seg[z]

            if fg.any():
                ys, xs = np.where(fg)
                cy = int(np.round(np.mean(ys)))
                cx = int(np.round(np.mean(xs)))
            else:
                cy, cx = H // 2, W // 2

            h_half = int(H * zoom_fraction / 2)
            w_half = int(W * zoom_fraction / 2)

            y0 = max(0, cy - h_half)
            y1 = min(H, cy + h_half)
            x0 = max(0, cx - w_half)
            x1 = min(W, cx + w_half)

        else:
            y0, y1 = 0, H
            x0, x1 = 0, W

        img_crop = img[z][y0:y1, x0:x1]
        gt_crop = gt[z][y0:y1, x0:x1]
        consensus_crop = consensus[z][y0:y1, x0:x1]

        recontour_crops = [
            r[z][y0:y1, x0:x1]
            for r in recontours
        ]

        fig, ax = plt.subplots(figsize=figsize)
        ax.imshow(img_crop, cmap="gray", zorder=0)

        # Draw ground truth first
        if show_ground_truth:
            draw_contour(
                ax,
                gt_crop,
                color_gt,
                linewidth=2,
                zorder=2,
            )

        # Draw observers second
        for i, (obs_crop, show_obs) in enumerate(
            zip(recontour_crops, show_observers)
        ):
            if show_obs:
                draw_contour(
                    ax,
                    obs_crop,
                    observer_colors[i],
                    linewidth=2,
                    zorder=3,
                )

        # Draw consensus LAST so it is always on top
        if show_consensus:
            draw_contour(
                ax,
                consensus_crop,
                color_consensus,
                linewidth=2,
                zorder=10,
            )

        handles = []

        if show_ground_truth:
            handles.append(
                mpatches.Patch(
                    color=color_gt,
                    label="Ground truth",
                )
            )

        if show_consensus:
            handles.append(
                mpatches.Patch(
                    color=color_consensus,
                    label="Consensus",
                )
            )

        for i, (obs_name, show_obs) in enumerate(
            zip(observer_names, show_observers)
        ):
            if show_obs:
                handles.append(
                    mpatches.Patch(
                        color=observer_colors[i],
                        label=f"Observer {obs_name}",
                    )
                )

        if handles:
            ax.legend(
                handles=handles,
                loc="upper right",
                framealpha=0.9,
            )

        ax.set_title(f"Slice {z}")
        ax.set_axis_off()

        plt.tight_layout()
        plt.show()

    interact(
        _plot,
        slice_idx=IntSlider(
            min=0,
            max=n_slices - 1,
            value=n_slices // 2,
            description="slice",
        ),
        zoom=Checkbox(
            value=False,
            description="zoom",
        ),
        show_ground_truth=Checkbox(
            value=True,
            description="ground truth",
        ),
        show_consensus=Checkbox(
            value=True,
            description="consensus",
        ),
        show_observer_B=Checkbox(
            value=True,
            description="observer B",
        ),
        show_observer_C=Checkbox(
            value=True,
            description="observer C",
        ),
        show_observer_D=Checkbox(
            value=True,
            description="observer D",
        ),
        show_observer_E=Checkbox(
            value=True,
            description="observer E",
        ),
    )

In [ ]:
data.load_recontours()
data.load_consensus()

plot_consensus_interactive(data)

interactive(children=(IntSlider(value=44, description='slice', max=87), Checkbox(value=False, description='zoo…